# Análisis Estadístico — Auditoría E-Gov Suroccidente de Guatemala

Notebook tutorial para explorar de forma interactiva los resultados del **estudio
longitudinal**. Consume la **tabla consolidada** (1 fila por portal), que es la
unidad de análisis de la tesis.

> **Anti-pseudoreplicación:** las mediciones repetidas (varias corridas por
> portal) se reducen a un único registro por portal *antes* de cualquier
> inferencia. Este notebook nunca analiza los snapshots crudos repetidos.

**Prerrequisito:** haber recolectado datos (`python run_daily.py` o GitHub
Actions) y, opcionalmente, haber corrido `python analizar.py`.


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

from src.analysis import stats as A
from src.collect.store import cargar_snapshots_df, resumen_cobertura
from src.consolidate.consolidator import consolidar, a_formato_reporte

pd.set_option('display.max_columns', 80)
sns.set_style('whitegrid')

## 0. Carga de datos

Usa `consolidado_latest.csv` si existe; si no, consolida al vuelo desde los snapshots de `data/daily/`.

In [ ]:
consol_csv = Path('../data/consolidated/consolidado_latest.csv')

if consol_csv.exists():
    consol = pd.read_csv(consol_csv)
    fuente = 'CSV consolidado'
else:
    snaps = cargar_snapshots_df()
    consol = consolidar(snaps) if not snaps.empty else pd.DataFrame()
    fuente = 'consolidado al vuelo' if not consol.empty else 'SIN DATOS'

print('Fuente:', fuente)
print('Cobertura acumulada:', resumen_cobertura())
assert not consol.empty, 'No hay datos. Corre run_daily.py / analizar.py primero.'

# df: consolidado con alias compatibles (score_local_*, ssl_ok, laip_*, ...)
df = a_formato_reporte(consol)
print(f'Portales: {len(df)} | columnas: {len(df.columns)}')
df[['municipio', 'departamento', 'tipo_hosting', 'uptime_pct',
    'cumple_LAIP', 'tiene_vulnerabilidad']].head(10)

## 1. Cobertura y disponibilidad (uptime)

`uptime_pct` = % de corridas en que el portal respondió, sobre el total de intentos.

In [ ]:
display(A.descriptiva_numerica(df, ['n_corridas', 'n_exitosas', 'uptime_pct']))

fig, ax = plt.subplots(figsize=(9, 5))
sub = df.sort_values('uptime_pct')
ax.barh(sub['municipio'], sub['uptime_pct'], color='#1F4E78')
ax.set_xlabel('Uptime (%)')
ax.set_xlim(0, 100)
ax.set_title('Disponibilidad por portal')
plt.tight_layout()
plt.show()

## 2. OE1 — Rendimiento y accesibilidad

Medianas por portal (sobre corridas exitosas).

In [ ]:
display(A.descriptiva_numerica(df, ['ttfb_ms', 'tiempo_total_ms',
                                    'tamanio_kb', 'score_local_performance']))
A.proporciones(df, ['https', 'tiene_viewport'])

## 3. OE2 — Transparencia LAIP (Decreto 57-2008)

In [ ]:
display(A.descriptiva_numerica(df, ['laip_pct_cumplimiento', 'score_local_freshness']))

laip_cols = ['laip_transparencia', 'laip_presupuesto', 'laip_compras',
             'laip_personal', 'laip_servicios', 'laip_estructura', 'laip_contacto']
A.proporciones(df, [c for c in laip_cols if c in df.columns])

## 4. OE3 — Seguridad básica

Incluye la distribución del estado SSL modal por portal.

In [ ]:
display(A.proporciones(df, ['ssl_ok', 'redirige_a_https', 'header_hsts',
                            'header_csp', 'header_x_frame_options']))

if 'ssl_estado_modal' in df.columns:
    display(df['ssl_estado_modal'].value_counts())

A.descriptiva_numerica(df, ['score_local_security'])

## 5. Comparación entre departamentos (Kruskal-Wallis)

¿Hay diferencias significativas (α=0.05) entre departamentos en cada métrica?

In [ ]:
A.comparar_departamentos_kruskal(df, [
    'tiempo_total_ms', 'score_local_performance',
    'laip_pct_cumplimiento', 'score_local_security', 'uptime_pct',
])

## 6. Correlaciones entre los ejes (Spearman)

¿Los portales con mejor rendimiento son también más transparentes y seguros?

In [ ]:
cols_corr = [c for c in ['score_local_performance', 'score_local_freshness',
             'score_local_security', 'laip_pct_cumplimiento', 'uptime_pct']
             if c in df.columns]
corr = A.correlaciones(df, cols_corr)
fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(corr, annot=True, cmap='RdBu_r', center=0, ax=ax, fmt='.2f')
ax.set_title('Correlaciones Spearman entre los ejes')
plt.tight_layout()
plt.show()

## 7. OE4 — Análisis de datos categóricos

Responde a la **pregunta auxiliar 4**: ¿existen asociaciones estadísticamente
significativas (α = 0.05) entre el cumplimiento del Decreto 57-2008 y las
variables operativas (departamento, cabecera, calidad técnica, tipo de hosting)?

Variables dependientes (ya derivadas en la consolidación):
- `cumple_LAIP` (1 = publica todos los apartados obligatorios)
- `tiene_vulnerabilidad` (1 = SSL inválido/autofirmado/mismatch, o sin HTTPS forzado, o sin HSTS+XFO+CSP)


In [ ]:
oe4 = A.analisis_oe4_completo(df)
print('Umbral LAIP usado:', oe4['umbral_laip_usado'])

### 7.1. Chi-cuadrado / Fisher — Cumplimiento LAIP vs predictores

Las filas con `significativo_alfa005 = True` indican asociación (p < 0.05). `V_cramer` mide la magnitud.

In [ ]:
oe4['chi2_laip']

### 7.2. Chi-cuadrado / Fisher — Vulnerabilidad vs predictores

In [ ]:
oe4['chi2_vuln']

### 7.3. Regresión logística binaria — Cumplimiento LAIP

Odds ratios (OR) con IC95%. Nivel positivo = *Cumple*.

In [ ]:
print('Metricas del modelo LAIP:')
display(pd.Series(oe4['logit_laip_metricas']))
oe4['logit_laip_coef']

### 7.4. Regresión logística binaria — Vulnerabilidad

Nivel positivo = *Vulnerable*.

In [ ]:
print('Metricas del modelo Vulnerabilidad:')
display(pd.Series(oe4['logit_vuln_metricas']))
oe4['logit_vuln_coef']

### 7.5. Tablas de contingencia (predictor × Cumplimiento LAIP)

In [ ]:
for predictor, tabla in oe4['tablas_contingencia_laip'].items():
    print(f'\n=== {predictor} ===')
    display(tabla)

---
### Notas de interpretación
- **Chi-cuadrado vs Fisher:** cuando alguna frecuencia esperada es < 5, la columna
  `prueba_recomendada` usa Fisher (tablas 2×2) o Monte Carlo (R×C); de lo
  contrario usa chi-cuadrado de independencia.
- **V de Cramér** (Cohen): 0.10 pequeña · 0.30 mediana · 0.50 grande.
- **Odds ratio (OR):** > 1 aumenta las probabilidades del evento; < 1 las reduce.
- Con n ≈ 39 portales, algunos modelos pueden no converger o tener IC amplios;
  es esperable y debe reportarse con cautela en la tesis.
